# 03. Pattern Matching

**목표**: `echo_tof.pattern` + `echo_tof.filters` 모듈을 이해한다.

Formula Enumeration(01)에서 후보를 생성하고, Isotope Calculator(02)로 이론 패턴을 계산했다면,
이제 두 가지 작업이 남는다:

1. **Pattern Matching**: 이론 vs 실측 동위원소 패턴 비교 (ClusterError, RMSError)
2. **5-Stage Filter**: 화학적 규칙으로 비합리적 후보 제거

이 두 모듈은 원본의 `MoleculePattern`과 `MonoMassResults`를 이식한 것이다.

## 환경 설정

In [ ]:
import os, sys
os.chdir(r'C:\Users\gogoc\AppData\Local\Temp\echo-tof-verify')
sys.path.insert(0, '.')

## Part A: Pattern Matching (`pattern.py`)

### 1. MoleculePattern 클래스

`MoleculePattern`은 분자의 이론적 isotope pattern을 생성하고,
실측 m/z + intensity 배열과 비교하여 두 가지 오차를 계산한다:

| 오차 | 의미 | 계산 방식 |
|------|------|----------|
| **ClusterError** | 강도(intensity) 일치도 | 이론 vs 실측 상대강도 차이의 RMS |
| **RMSError** | 질량(mass) 일치도 | 이론 vs 실측 m/z 오차의 RMS (ppm) |

### 2. ClusterError 계산

```python
def get_cluster_error(self, measured_mz, norm_measured_int, use_peak, sn_correction):
    # 최대 강도 피크는 제외 (기준 피크이므로)
    max_idx = argmax(norm_measured_int)
    
    for i in range(len(norm_measured_int)):
        if i == max_idx or not use_peak[i]:
            continue
        theor_int = self._result[best_idx].normalised_abundance * 100.0
        diff = abs(theor_int - norm_measured_int[i]) * sn_correction[i]
        sum_sq += diff ** 2
        norm_sq += norm_measured_int[i] ** 2
    
    return sqrt(sum_sq / norm_sq)
```

**핵심**: 기준 피크(최대 강도)는 항상 100%로 정규화되므로 비교에서 제외한다.
`sn_correction`은 S/N이 낮은 피크에 낮은 가중치를 부여한다.

### 3. RMSError 계산

```python
def get_rms_error(self, measured_mz, use_peak, sn_correction, adjust_mono_mz=False):
    # adjust_mono_mz=True: mono 피크 m/z 오차를 전체에서 빼서 보정
    if adjust_mono_mz:
        mono_offset = theor_mz[0] - measured_mz[0]
    
    for i in range(start, len(measured_mz)):
        error = get_mass_error(theor_mz, measured_mz[i] + mono_offset, in_ppm=True)
        error *= sn_correction[i]
        sum_sq += error ** 2
        count += 1
    
    return sqrt(sum_sq / count)
```

`adjust_mono_mz=True`는 TOF의 systematic mass shift를 보정하기 위한 옵션이다.

### 4. 실습: 이론 패턴 생성 및 오차 계산

In [ ]:
from echo_tof.pattern import MoleculePattern
from echo_tof.molecule import Molecule
from echo_tof.isotope_calc import IsotopicDistributionCalculator

# TNT 이론 패턴
mol = Molecule('C7H5N3O6', charge=0)
mp = MoleculePattern(charge=0)
idc = IsotopicDistributionCalculator(ppm_tolerance=True, tolerance=50.0)
mp.calculate_pattern(mol, idc)

print("이론적 m/z:")
for i, (mz, inten) in enumerate(zip(mp.pattern_mz, mp.pattern_rel_intensities)):
    label = f"M+{i}" if i > 0 else "M  "
    print(f"  {label}  m/z={mz:.4f}  intensity={inten:.2f}%")

In [ ]:
# 가상의 실측 데이터 (약간의 오차 포함)
measured_mz  = [227.0180, 228.0213, 229.0240]
measured_int = [100.0, 8.5, 2.0]  # 정규화된 %
use_peak     = [True, True, True]
sn_weight    = [1.0, 1.0, 1.0]

cluster_err = mp.get_cluster_error(measured_mz, measured_int, use_peak, sn_weight)
rms_err     = mp.get_rms_error(measured_mz, use_peak, sn_weight, adjust_mono_mz=False)
rms_w_int   = mp.get_rms_error(measured_mz, use_peak, sn_weight, adjust_mono_mz=True)

print(f"Cluster Error (intensity): {cluster_err:.4f}")
print(f"RMS Error (mass, ppm):     {rms_err:.4f}")
print(f"RMS w/ int correction:     {rms_w_int:.4f}")

---

## Part B: 5-Stage Filter Pipeline (`filters.py`)

`FormulaFilter`는 Formula Enumeration의 결과를 5단계로 걸러낸다.
각 단계는 독립적인 화학적 규칙을 적용한다.

### Stage 1: `_fits_rdb()` - DBE 범위

**DBE** (Double Bond Equivalents, Ring + Double Bond)가 지정 범위 내인지 확인.

```python
def _fits_rdb(self, mol, max_rdb):
    return self._dbe_from <= mol.rdb <= max_rdb
```

기본값: -0.5 ~ 40.0. 유기 분자는 보통 RDB >= 0.

### Stage 2: `_fits_electron_state()` - 전자 상태

짝수/홀수 전자 분자 필터. [M+H]+는 even-electron, [M]+. 는 odd-electron.

### Stage 3: `_fits_hetero_atoms()` - 헤테로원자 비율

- C/hetero 비율 최소값
- O/S, O/P 비율 (옵션)

### Stage 4: `_fits_multiple_element_rules()` - 다원소 조합 규칙

N, S, P, O가 동시에 많이 존재하면 비현실적인 분자. 예:

```python
# N>1, S>1, O>1, P>1 이면: N<10, O<20, P<4, S<3 이어야 함
if n > 1 and s > 1 and o > 1 and p > 1:
    if not (n < 10 and o < 20 and p < 4 and s < 3):
        return False
```

### Stage 5: `_fits_element_ratio_rules()` - 원소비율 규칙

| 비율 | 허용 범위 |
|------|----------|
| H/C | 0.2 ~ 3.1 |
| N/C | < 1.3 |
| O/C | < 1.2 |
| S/C | < 0.8 |
| F/C | < 1.5 |

이 규칙들은 Kind & Fiehn (2007)의 "Seven Golden Rules"에 기반한다.

### 5. 실습: 필터 파이프라인 적용

MW=227.016 (TNT 부근)에 대해 열거 + 필터링을 수행한다.

In [ ]:
from echo_tof.filters import FormulaFilter
from echo_tof.formula_enum import find_compositions

# 먼저 열거만 수행 (필터 없이)
raw_results = find_compositions(
    target_mass=227.016,
    mass_tolerance=227.016 * 5.0 / 1e6,  # 5 ppm
    max_composition="C50 H200 N10 O10 S5",
)
print(f"필터 전 후보 수: {len(raw_results)}")

In [ ]:
# FormulaFilter로 5단계 필터링
ff = FormulaFilter(
    mass_tolerance_ppm=5.0,
    dbe_from=-0.5,
    dbe_to=40.0,
    even_electron=True,
    odd_electron=True,
    common_rules=True,  # Stage 4, 5 활성화
)

filtered = ff.get_compositions(
    min_composition="",
    max_composition="C50 H200 N10 O10 S5",
    mono_mz=227.016,
    charge=0,
)

print(f"필터 후 후보 수: {len(filtered)}")
print()
print(f"{'Formula':20s} {'Mass':>12s} {'Error(mDa)':>10s} {'RDB':>6s} {'Even-e':>7s}")
print("-" * 58)
for mol in filtered[:15]:
    err = (mol.monoisotopic_mass - 227.016) * 1000
    print(f"{mol.composition:20s} {mol.monoisotopic_mass:12.6f} {err:+10.3f} "
          f"{mol.rdb:6.1f} {'Y' if mol.is_even_electron else 'N':>7s}")

### 6. 필터별 제거 효과 확인

`common_rules=False`로 Stage 4, 5를 비활성화하면 얼마나 차이나는지 확인한다.

In [ ]:
ff_no_rules = FormulaFilter(
    mass_tolerance_ppm=5.0,
    even_electron=True,
    odd_electron=True,
    common_rules=False,  # Stage 4, 5 비활성화
)

no_rules = ff_no_rules.get_compositions("", "C50 H200 N10 O10 S5", 227.016, 0)
print(f"common_rules=True:  {len(filtered)} candidates")
print(f"common_rules=False: {len(no_rules)} candidates")
print(f"제거된 후보: {len(no_rules) - len(filtered)}")

## 정리

| 모듈 | 클래스 | 역할 |
|------|--------|------|
| `pattern.py` | `MoleculePattern` | 이론 패턴 생성, ClusterError/RMSError 계산 |
| `filters.py` | `FormulaFilter` | 5단계 화학 규칙 필터링 |

**5-Stage Filter Pipeline**:
1. DBE 범위 체크
2. 전자 상태 (even/odd)
3. 헤테로원자 비율 (C/hetero, O/S, O/P)
4. 다원소 조합 규칙 (NOPS 동시 존재 제한)
5. 원소비율 규칙 (H/C, N/C, O/C 등)

다음 노트북에서는 이 모든 것을 통합한 **FormulaFinderPipeline**을 다룬다.